#### Key code you will work with:
- This example shows you how to track token usage for cost estimation and monitoring, helping you optimize your AI application expenses.

In [ ]:
# Make a request
response = model.invoke("Explain what Python is in 2 sentences.")

# Extract token usage from the response metadata
usage = response.usage_metadata

print(f"  Prompt tokens:     {usage.get('input_tokens')}")   # Your input
print(f"  Completion tokens: {usage.get('output_tokens')}")  # AI's response
print(f"  Total tokens:      {usage.get('total_tokens')}")   # Total cost basis

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain.chat_models import init_chat_model

def track_token_usage():
        model = init_chat_model(
            "openai/gpt-oss-120b",
            model_provider="groq",
            temperature=0
        )

        
        # Make a request
        response = model.invoke("Explain what is Python in 2 sentences")

        # Extract token usage from metadata
        usage = response.usage_metadata


        if usage:
            print("Token Breakdown:")
            print(f"  Prompt tokens:     {usage.get('input_tokens', 'N/A')}")
            print(f"  Completion tokens: {usage.get('output_tokens', 'N/A')}")
            print(f"  Total tokens:      {usage.get('total_tokens', 'N/A')}")
        else:
            print(" Token usage information not available in response metadata.")

        print("\nResponse:")
        print(response.content)

if __name__ == "__main__":
        track_token_usage()



Token Breakdown:
  Prompt tokens:     79
  Completion tokens: 103
  Total tokens:      182

Response:
Python is a high‑level, interpreted programming language known for its clear syntax and readability, which makes it ideal for rapid development across a wide range of applications—from web development and data analysis to artificial intelligence and scientific computing. Its extensive standard library and vibrant ecosystem of third‑party packages enable developers to write powerful code with relatively few lines.


How can I track token usage across multiple API calls in a conversation?
How would I calculate the cost based on token usage?

In [12]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv

load_dotenv()

# Prices per 1 million tokens for openai/gpt-oss-120b
INPUT_PRICE_PER_MILLION = 0.15
OUTPUT_PRICE_PER_MILLION = 0.60


def calculate_cost(input_tokens, output_tokens):
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    return input_cost + output_cost


def track_conversation_usage():
    model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0)

    messages = [SystemMessage(content="You are a helpful coding tutor. Keep answers under 100 words.")]
    questions = [
        "What is Python?",
        "Can you show me a simple example?",
        "What are its benefits compared to other languages?"
    ]

    total_input = 0
    total_output = 0

    for i, question in enumerate(questions, start=1):
        messages.append(HumanMessage(content=question))
        response = model.invoke(messages)
        messages.append(AIMessage(content=response.content))

        usage = response.usage_metadata
        total_input += usage["input_tokens"]
        total_output += usage["output_tokens"]

        print(f"Call {i}: input={usage['input_tokens']}, output={usage['output_tokens']}")

    # These lines are INSIDE the function (indented), so they can see the totals
    print(f"\nTotal input tokens:  {total_input}")
    print(f"Total output tokens: {total_output}")
    print(f"Total tokens:        {total_input + total_output}")

    cost = calculate_cost(total_input, total_output)
    print(f"Estimated cost: ${cost:.6f}")


if __name__ == "__main__":
    track_conversation_usage()

Call 1: input=92, output=125
Call 2: input=204, output=155
Call 3: input=333, output=254

Total input tokens:  629
Total output tokens: 534
Total tokens:        1163
Estimated cost: $0.000415


The key rule: the print and cost lines must be indented under def track_conversation_usage():, at the same level as the for loop. Indented means inside the function, where total_input exists. At the left edge means outside, where it doesn’t.